# SmolVLM PosPrune with per-camera smoothing splines

Select **GPU T4 ×2**, enable Internet, and use **Save & Run All**. The notebook downloads the pinned checkpoint and data, then extracts temporary vision features. No Kaggle Input is required.

For each of the six cameras, fit a natural cubic smoothing spline on its 64 positions using the mean attention difference from block 2 to block 23 (`layers[1]` minus `layers[22]`). Fit on 800 validation questions. Subtract the fitted value at each visual position from block-2 attention. Evaluate residuals separately on the same 4,204 stratified test questions as the recent MMR notebooks. The smoothing parameters were selected with recording-wise cross-validation on these 800 validation questions in the local CUDA run. The report compares held-out RMSE with per-camera and global cubic fits.

All four methods prune immediately after block 2, keeping 32 or 64 visual tokens across the six cameras. Raw and corrected top-k use their respective attention scores. Raw + MMR and corrected + MMR use the corresponding scores after per-question min–max normalization over all 384 tokens, with block-2 output hidden-state cosine for redundancy and λ = 0.9. The eight configurations use FP32 greedy decoding. Deltas use raw attention at the same budget as the reference.

In [ ]:
GPU_COUNT = 2
CALIBRATION_SIZE = 800
CALIBRATION_BATCH = 4  # Per GPU.
GENERATION_BATCH = 4   # Per GPU, with automatic CUDA OOM backoff.
KEEP_COUNTS = (32, 64)
CAMERA_BIAS_LAYERS = (2, 23)  # One-based decoder block numbers.
MMR_LAMBDA = 0.9
EVALUATION_SUBSET_SIZE = 4204
SELECTION_EXAMPLES = 3  # Save full ranked selections for the first evaluation examples.
CACHE_INPUT_DIR = ''   # Leave blank to fetch images and build features automatically.
# Fixed by recording-wise validation on the unit-spaced camera grid.
SPLINE_SMOOTHING = {
    'Front': 0.01,
    'Front-Left': 1000.0,
    'Front-Right': 100.0,
    'Back': 0.01,
    'Back-Left': 10000.0,
    'Back-Right': 3.0,
}

In [ ]:
# Embedded source snapshot. Run this cell before importing Transformers.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
from pathlib import Path
from base64 import b64decode
from io import BytesIO
import shutil
import subprocess
import sys
import tempfile
import zipfile

if shutil.which('java') is None:
    raise RuntimeError('Java is required by the existing COCO caption evaluation metrics.')
SOURCE = Path(tempfile.mkdtemp(prefix='log-metrics-posprune-'))
SOURCE_B64 = '...'
with zipfile.ZipFile(BytesIO(b64decode(SOURCE_B64))) as archive:
    archive.extractall(SOURCE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                '-r', str(SOURCE / 'smolvlm_attention/requirements.txt'), 'pycocoevalcap==1.2'], check=True)
sys.path.insert(0, str(SOURCE))

In [ ]:
import torch
import transformers
import tokenizers
from huggingface_hub import hf_hub_download
from smolvlm_attention.config import MODELS, CACHE_DATASET, CACHE_FINGERPRINT, PREPARED_REPO, PREPARED_REVISION
from smolvlm_attention.posprune_ablation.cache_input import resolve_cache_input

if (transformers.__version__, tokenizers.__version__) != ('4.46.3', '0.20.4'):
    raise RuntimeError('Restart the kernel after dependency installation and Run All.')
visible_gpus = torch.cuda.device_count()
if visible_gpus == 0:
    raise RuntimeError('Enable a Kaggle GPU accelerator.')
if GPU_COUNT < 1:
    raise ValueError('GPU_COUNT must be at least 1.')
worker_count = min(GPU_COUNT, visible_gpus)
print(f'Using {worker_count} of {visible_gpus} visible GPU(s).')

cache_dir = None
if CACHE_INPUT_DIR:
    cache_dir = resolve_cache_input('/kaggle/input', CACHE_INPUT_DIR, dataset=CACHE_DATASET,
                                   fingerprint=CACHE_FINGERPRINT)
    print(f'Using mounted feature cache: {cache_dir}')
work = Path('/kaggle/working/smolvlm-posprune-camera-spline-2-23')
prepared_dir = work / 'prepared_data'
for name in ('manifest.json', 'train.jsonl', 'val.jsonl', 'test.jsonl'):
    hf_hub_download(repo_id=PREPARED_REPO, repo_type='dataset', revision=PREPARED_REVISION,
                    filename=name, local_dir=prepared_dir, token=False)
base = MODELS['base']
checkpoint = hf_hub_download(repo_id=base['repo'], revision=base['revision'],
                             filename='finetune_best.pt', token=False)

In [ ]:
import json
cohort = json.loads((SOURCE / 'camera_posprune_cohort.json').read_bytes())

from smolvlm_attention.posprune_ablation.parallel import run_parallel

run_dir = Path(tempfile.mkdtemp(prefix='run-', dir=work))
output = run_parallel(checkpoint, cache_dir, prepared_dir, run_dir / 'results',
                      cache_fingerprint=CACHE_FINGERPRINT, camera_order=base['camera_order'], gpu_count=worker_count,
                      calibration_size=CALIBRATION_SIZE, evaluation_size=None, evaluation_fraction=cohort['evaluation_fraction'],
                      calibration_batch=CALIBRATION_BATCH, generation_batch=GENERATION_BATCH,
                      keep_counts=KEEP_COUNTS, seed=cohort['seed'],
                      selection_examples=SELECTION_EXAMPLES, camera_bias_layers=CAMERA_BIAS_LAYERS,
                      mmr_weight=MMR_LAMBDA, attention_normalization='minmax',
                      evaluation_subset_size=EVALUATION_SUBSET_SIZE, expected_evaluation_ids=cohort['evaluation_ids'],
                      spline_smoothing=SPLINE_SMOOTHING)
print((output / 'parallel_timing.json').read_text())
print(output)

In [ ]:
from IPython.display import FileLink, Image, display
from smolvlm_attention.posprune_ablation.report import report

for path in report(output):
    if path.suffix == '.png':
        display(Image(filename=str(path)))
archive = run_dir / 'posprune-results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(output.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
tables = [output / name for name in ('indexed_selections.csv', 'comparison.csv', 'paired_answers.csv',
                                      'question_type_distribution.csv')]
for path in (*tables, archive):
    display(FileLink(os.path.relpath(path)))

display(FileLink(os.path.relpath(output / 'camera_fit_metrics.csv')))